# 06 — Cenários de política e recomendação final

**Objetivo:** Comparar cenários de política de crédito e recomendar uma política final de concessão de empréstimo, com base no trade-off entre inadimplência dos aprovados, taxa de aprovação, análise manual, recusa e exposição financeira aprovada.

**Entrada:** `data/processed/base_politica_validacao_com_score.parquet` — safra de validação com score de PD e política inicial criados no notebook 05.

**Produto:** Empréstimo bancário parcelado para pessoa física com relacionamento bancário.

## 1. Contexto de negócio e premissas da política

### Enquadramento do produto

O case trata de uma política de **concessão de empréstimo bancário parcelado** para **pessoa física com relacionamento bancário**. A base contém dados disponíveis no momento da proposta: renda, valor solicitado, taxa, prazo, restritivos e relacionamento com o banco.

### O que esta política não é

- Não é política de abertura de conta
- Não é política PJ (sem faturamento, CNAE, porte ou balanço)
- Não é política baseada em score externo de bureau
- Não é política com garantia formal conhecida
- Não é política completa com LGD e EAD regulatório

### Score e rating interno

O `pd_score` foi construído por modelagem supervisionada no notebook 04 e representa a **probabilidade estimada de inadimplência em 12 meses**. No notebook 05, esse score foi convertido em **faixas de risco (A a E)**, que funcionam como rating interno de crédito.

### Variáveis de decisão

A política combina:
- **Risco estimado:** `pd_score`, `faixa_risco`
- **Capacidade de pagamento:** `valor_renda`, `comprometimento_renda`
- **Restritivos financeiros:** `valor_restritivos`, `restritivos_sobre_renda`
- **Relacionamento:** `flag_cliente_ativo`, `tempo_conta_anos`
- **Características da operação:** `valor_emprestado`, `valor_taxa`, `valor_prazo`

### Variável de avaliação (somente backtest)

`target_inadimplente_12m` — **nunca usada como variável de decisão**, apenas para avaliar o desempenho histórico das políticas simuladas.

### Decisões possíveis

| Decisão | Descrição |
|---|---|
| Aprovar valor solicitado | Limite calculado cobre o valor pedido |
| Aprovar valor reduzido | Limite calculado é menor que o pedido, mas operacionalmente viável |
| Análise manual | Risco elevado ou incerteza — requer revisão humana |
| Recusar | Risco muito alto ou capacidade insuficiente |

## 2. Setup e leitura da base com score

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.concessao_credito.config import PROCESSED_DIR, TABLES_DIR, FIGURES_DIR
from src.concessao_credito.plots import (
    registrar_template_credito,
    aplicar_layout,
    CORES,
)

registrar_template_credito()

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)

print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"TABLES_DIR    : {TABLES_DIR}")
print(f"FIGURES_DIR   : {FIGURES_DIR}")

PROCESSED_DIR : C:\GitHub\datascience\projetos\concessao_credito\data\processed
TABLES_DIR    : C:\GitHub\datascience\projetos\concessao_credito\outputs\tables
FIGURES_DIR   : C:\GitHub\datascience\projetos\concessao_credito\outputs\figures


In [2]:
CAMINHO_BASE = PROCESSED_DIR / "base_politica_validacao_com_score.parquet"

if not CAMINHO_BASE.exists():
    raise FileNotFoundError(
        f"Base principal não encontrada: {CAMINHO_BASE}\n"
        "Execute o notebook 05 antes de continuar."
    )

df = pd.read_parquet(CAMINHO_BASE)

print(f"Base carregada com sucesso.")
print(f"  Linhas  : {df.shape[0]:,}")
print(f"  Colunas : {df.shape[1]}")

Base carregada com sucesso.
  Linhas  : 4,862
  Colunas : 41


## 3. Contrato de dados e validações iniciais

In [3]:
COLUNAS_OBRIGATORIAS = [
    "id_cliente", "pd_score", "faixa_risco",
    "valor_renda", "valor_emprestado", "valor_parcela",
    "valor_taxa", "valor_prazo", "valor_restritivos",
    "restritivos_sobre_renda", "comprometimento_renda",
    "tempo_conta_anos", "flag_cliente_ativo", "target_inadimplente_12m",
]

colunas_faltantes = [c for c in COLUNAS_OBRIGATORIAS if c not in df.columns]
if colunas_faltantes:
    raise ValueError(f"Colunas obrigatorias ausentes: {colunas_faltantes}")
print("✓ Todas as colunas obrigatorias presentes.")

# id_operacao
if "id_operacao" not in df.columns:
    df["id_operacao"] = df.index + 1
    print("  id_operacao criado a partir do indice (cada cliente aparece uma vez).")
else:
    print(f"✓ id_operacao encontrado na base ({df['id_operacao'].nunique():,} valores unicos).")

# pd_score
score_min = df["pd_score"].min()
score_max = df["pd_score"].max()
if not df["pd_score"].between(0, 1).all():
    raise ValueError(f"pd_score fora de [0,1]: min={score_min:.4f}, max={score_max:.4f}")
print(f"✓ pd_score em [{score_min:.4f}, {score_max:.4f}]")

# faixa_risco
FAIXAS_ESPERADAS = {
    "A - Baixo risco", "B - Médio-baixo risco", "C - Médio risco",
    "D - Alto risco", "E - Muito alto risco",
}
faixas_encontradas = set(df["faixa_risco"].unique())
faixas_invalidas = faixas_encontradas - FAIXAS_ESPERADAS
if faixas_invalidas:
    print(f"Faixas nao esperadas: {faixas_invalidas}")
else:
    print(f"✓ faixa_risco: {len(faixas_encontradas)} categorias validas")

# taxa historica
TAXA_HISTORICA = df["target_inadimplente_12m"].mean()
print(f"✓ Taxa historica de inadimplencia: {TAXA_HISTORICA:.2%} "
      f"({int(df['target_inadimplente_12m'].sum())} de {len(df):,} operacoes)")

✓ Todas as colunas obrigatorias presentes.
✓ id_operacao encontrado na base (4,862 valores unicos).
✓ pd_score em [0.0047, 0.9214]
✓ faixa_risco: 5 categorias validas
✓ Taxa historica de inadimplencia: 12.30% (598 de 4,862 operacoes)


In [4]:
# Tabela de distribuição das faixas
df_dist_faixas = (
    df.groupby("faixa_risco", observed=True)
    .agg(
        qtd_operacoes=("id_operacao", "count"),
        pd_medio=("pd_score", "mean"),
        taxa_inadimplencia=("target_inadimplente_12m", "mean"),
    )
    .reset_index()
)

ordem_f = ["A - Baixo risco", "B - Médio-baixo risco", "C - Médio risco", "D - Alto risco", "E - Muito alto risco"]
df_dist_faixas["faixa_risco"] = pd.Categorical(df_dist_faixas["faixa_risco"], categories=ordem_f, ordered=True)
df_dist_faixas = df_dist_faixas.sort_values("faixa_risco")
df_dist_faixas["participacao"] = df_dist_faixas["qtd_operacoes"] / df_dist_faixas["qtd_operacoes"].sum()

print("Distribuicao da base por faixa de risco:")
df_dist_faixas

Distribuicao da base por faixa de risco:


,faixa_risco,qtd_operacoes,pd_medio,taxa_inadimplencia,participacao
0,A - Baixo risco,1945,0.0225,0.0108,0.4000
1,B - Médio-baixo risco,972,0.0607,0.0453,0.1999
2,C - Médio risco,729,0.1098,0.1056,0.1499
3,D - Alto risco,729,0.2253,0.2510,0.1499
4,E - Muito alto risco,487,0.5224,0.5606,0.1002


In [5]:
# Contrato de dados completo
papel_colunas = {
    "id_operacao":              ("identificador", "Identificador da operacao"),
    "id_cliente":               ("identificador", "Identificador do cliente"),
    "pd_score":                 ("risco", "Probabilidade de inadimplencia estimada pelo modelo"),
    "faixa_risco":              ("risco", "Rating interno discretizado do pd_score"),
    "valor_renda":              ("capacidade", "Base do calculo de parcela maxima"),
    "valor_emprestado":         ("operacao/exposicao", "Valor solicitado pelo cliente"),
    "valor_parcela":            ("operacao/capacidade", "Parcela original historica"),
    "valor_taxa":               ("operacao", "Taxa mensal usada no calculo de valor presente"),
    "valor_prazo":              ("operacao", "Prazo em meses"),
    "valor_restritivos":        ("restritivo", "Dividas em orgaos de protecao ao credito"),
    "restritivos_sobre_renda":  ("restritivo", "Proxy de pressao financeira - redutor de limite"),
    "comprometimento_renda":    ("capacidade", "Parcela/renda historica - diagnostico"),
    "tempo_conta_anos":         ("relacionamento", "Proxy de fidelidade com o banco"),
    "flag_cliente_ativo":       ("relacionamento", "1=ativo, 0=inativo"),
    "target_inadimplente_12m":  ("avaliacao", "SOMENTE backtest - nunca usar como feature de decisao"),
    "decisao_politica":         ("referencia", "Decisao da politica inicial do notebook 05"),
    "valor_aprovado_politica":  ("referencia", "Valor aprovado pela politica inicial"),
    "valor_maximo_politica":    ("referencia", "Limite maximo calculado pela politica inicial"),
}

registros_contrato = []
for col in df.columns:
    n_nulos = int(df[col].isna().sum())
    pct_nulos = n_nulos / len(df)
    papel, obs = papel_colunas.get(col, ("-", "-"))
    registros_contrato.append({
        "coluna": col,
        "tipo": str(df[col].dtype),
        "n_nulos": n_nulos,
        "pct_nulos": f"{pct_nulos:.1%}",
        "papel_na_politica": papel,
        "observacao": obs,
    })

df_contrato = pd.DataFrame(registros_contrato)
df_contrato.to_csv(TABLES_DIR / "politica_contrato_dados_cenarios.csv", index=False)
print(f"Contrato salvo -> {TABLES_DIR / 'politica_contrato_dados_cenarios.csv'}")
df_contrato

Contrato salvo -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_contrato_dados_cenarios.csv


,coluna,tipo,n_nulos,pct_nulos,papel_na_politica,observacao
0,id_cliente,int64,0,0.0%,identificador,Identificador do cliente
1,data_concessao,datetime64[us],0,0.0%,-,-
2,valor_emprestado,float64,0,0.0%,operacao/exposicao,Valor solicitado pelo cliente
3,valor_parcela,float64,0,0.0%,operacao/capacidade,Parcela original historica
4,valor_taxa,float64,0,0.0%,operacao,Taxa mensal usada no calculo de valor presente
5,valor_prazo,int64,0,0.0%,operacao,Prazo em meses
6,valor_renda,float64,0,0.0%,capacidade,Base do calculo de parcela maxima
7,valor_restritivos,float64,0,0.0%,restritivo,Dividas em orgaos de protecao ao credito
8,cat_escolaridade,string,0,0.0%,-,-
9,flag_cliente_ativo,int64,0,0.0%,relacionamento,"1=ativo, 0=inativo"


## 4. Diagnóstico da política inicial

A política inicial foi criada no notebook 05. Os resultados aqui servem como **linha de base** para comparação com os novos cenários.

In [6]:
COLUNAS_POLITICA_INICIAL = ["decisao_politica", "valor_aprovado_politica", "valor_maximo_politica"]
tem_politica_inicial = all(c in df.columns for c in COLUNAS_POLITICA_INICIAL)

if tem_politica_inicial:
    print("Politica inicial encontrada. Calculando indicadores...\n")

    mask_aprov_ini = df["decisao_politica"].isin(["Aprovar valor solicitado", "Aprovar valor reduzido"])
    val_orig_ini = df["valor_emprestado"].sum()
    val_aprov_ini = df["valor_aprovado_politica"].sum()

    diag_ini = {
        "Quantidade de operacoes":          len(df),
        "Taxa historica de inadimplencia":  f"{TAXA_HISTORICA:.2%}",
        "Taxa de aprovacao automatica":     f"{mask_aprov_ini.mean():.2%}",
        "Taxa de analise manual":           f"{(df['decisao_politica']=='Análise manual').mean():.2%}",
        "Taxa de recusa":                   f"{(df['decisao_politica']=='Recusar').mean():.2%}",
        "Inadimplencia dos aprovados":      f"{df.loc[mask_aprov_ini, 'target_inadimplente_12m'].mean():.2%}",
        "PD media dos aprovados":           f"{df.loc[mask_aprov_ini, 'pd_score'].mean():.4f}",
        "Valor original total":             f"R$ {val_orig_ini:,.0f}",
        "Valor aprovado total":             f"R$ {val_aprov_ini:,.0f}",
        "Exposicao aprovada":               f"{val_aprov_ini/val_orig_ini:.2%}",
        "Reducao de exposicao":             f"{1 - val_aprov_ini/val_orig_ini:.2%}",
        "Valor medio aprovado":             f"R$ {df.loc[mask_aprov_ini, 'valor_aprovado_politica'].mean():,.0f}",
    }

    df_diag_ini = pd.DataFrame.from_dict(diag_ini, orient="index", columns=["Politica inicial (NB 05)"])
    print(df_diag_ini.to_string())
    print("\nDistribuicao das decisoes da politica inicial:")
    print(df["decisao_politica"].value_counts().to_string())
else:
    print("Colunas da politica inicial nao encontradas. A politica inicial sera reconstruida como cenario de referencia.")

Politica inicial encontrada. Calculando indicadores...

                                Politica inicial (NB 05)
Quantidade de operacoes                             4862
Taxa historica de inadimplencia                   12.30%
Taxa de aprovacao automatica                      74.95%
Taxa de analise manual                            13.97%
Taxa de recusa                                    11.09%
Inadimplencia dos aprovados                        3.90%
PD media dos aprovados                            0.0501
Valor original total                       R$ 64,604,367
Valor aprovado total                       R$ 25,416,719
Exposicao aprovada                                39.34%
Reducao de exposicao                              60.66%
Valor medio aprovado                            R$ 6,975

Distribuicao das decisoes da politica inicial:
decisao_politica
Aprovar valor reduzido      2486
Aprovar valor solicitado    1158
Análise manual               679
Recusar                      539


## 5. Variáveis disponíveis e limitações da política

Esta seção mapeia variáveis típicas de uma política de crédito, sua disponibilidade no case e como serão usadas.

In [7]:
df_variaveis = pd.DataFrame([
    ("Renda PF",              "Sim",    "valor_renda",            "Base do calculo de parcela maxima",             "Autodeclarada; sem validacao externa"),
    ("Faturamento PJ",        "Nao",    "—",                      "Nao aplicavel — base e PF",                     "Base nao possui clientes PJ"),
    ("Ramo / CNAE",           "Nao",    "—",                      "Nao aplicavel — base e PF",                     "Base nao possui clientes PJ"),
    ("Score externo (bureau)","Nao",    "pd_score (interno)",     "Usado como rating interno de risco",            "Score criado no projeto; nao e bureau externo"),
    ("Garantia / colateral",  "Nao",    "—",                      "Nao disponivel",                                "Politica mais conservadora por ausencia de mitigador"),
    ("Historico de atraso",   "Parcial","valor_restritivos",      "Redutor de limite",                             "Proxy indireto; sem historico detalhado de atrasos"),
    ("Restritivos financeiros","Sim",   "restritivos_sobre_renda","Redutor multiplicativo de parcela maxima",      "Nivel relativo a renda"),
    ("Relacionamento bancario","Sim",   "flag_cliente_ativo",     "Redutor por cliente inativo",                   "Binario; sem profundidade transacional"),
    ("Tempo de conta",        "Sim",    "tempo_conta_anos",       "Redutor por relacionamento curto",              "Proxy de fidelidade; sem qualidade do relacionamento"),
    ("Segmento bancario",     "Nao",    "—",                      "Nao disponivel",                                "Sem segmentacao formal (varejo, private, etc.)"),
    ("Inadimplencia posterior","Sim",   "target_inadimplente_12m","SOMENTE avaliacao historica (backtest)",         "Nao pode ser usada como feature de decisao"),
    ("Propostas recusadas",   "Nao",    "—",                      "Nao disponivel",                                "Base contem apenas operacoes concedidas — vies de selecao"),
    ("LGD (perda dado default)","Nao",  "—",                      "Nao disponivel",                                "Sem informacao de recuperacao ou colateral"),
    ("EAD (exposicao no default)","Parcial","valor_emprestado",  "Proxy de exposicao",                            "Sem exposicao regulatoria formal"),
    ("Comprometimento de renda","Sim",  "comprometimento_renda",  "Diagnostico e validacao da politica",           "Calculado com a parcela historica concedida"),
], columns=[
    "variavel_tipica_politica", "existe_no_case", "variavel_equivalente",
    "como_sera_usada", "limitacao"
])

df_variaveis.to_csv(TABLES_DIR / "politica_variaveis_disponiveis_limitacoes.csv", index=False)
print(f"Tabela salva -> {TABLES_DIR / 'politica_variaveis_disponiveis_limitacoes.csv'}")
df_variaveis

Tabela salva -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_variaveis_disponiveis_limitacoes.csv


,variavel_tipica_politica,existe_no_case,variavel_equivalente,como_sera_usada,limitacao
0,Renda PF,Sim,valor_renda,Base do calculo de parcela maxima,Autodeclarada; sem validacao externa
1,Faturamento PJ,Nao,—,Nao aplicavel — base e PF,Base nao possui clientes PJ
2,Ramo / CNAE,Nao,—,Nao aplicavel — base e PF,Base nao possui clientes PJ
3,Score externo (bureau),Nao,pd_score (interno),Usado como rating interno de risco,Score criado no projeto; nao e bureau externo
4,Garantia / colateral,Nao,—,Nao disponivel,Politica mais conservadora por ausencia de mit...
5,Historico de atraso,Parcial,valor_restritivos,Redutor de limite,Proxy indireto; sem historico detalhado de atr...
6,Restritivos financeiros,Sim,restritivos_sobre_renda,Redutor multiplicativo de parcela maxima,Nivel relativo a renda
7,Relacionamento bancario,Sim,flag_cliente_ativo,Redutor por cliente inativo,Binario; sem profundidade transacional
8,Tempo de conta,Sim,tempo_conta_anos,Redutor por relacionamento curto,Proxy de fidelidade; sem qualidade do relacion...
9,Segmento bancario,Nao,—,Nao disponivel,"Sem segmentacao formal (varejo, private, etc.)"


## 6. Definição dos cenários de política

Três cenários de apetite de risco são simulados:

| Cenário | Objetivo | Percentuais máx. (A / B / C / D / E) |
|---|---|---|
| **Conservador** | Proteger carteira, reduzir inadimplência | 35% / 30% / 25% / Análise manual / Recusar |
| **Equilibrado** | Equilíbrio risco-volume | 40% / 35% / 30% / Análise manual / Recusar |
| **Expansivo** | Ampliar concessão, aceitar mais risco | 45% / 40% / 35% / 25% condicional / Recusar |

A **faixa E** é recusada em todos os cenários. A **faixa D** vai para análise manual nos cenários Conservador e Equilibrado; no Expansivo, pode ter aprovação reduzida se os restritivos forem aceitáveis (≤ 5% da renda).

In [8]:
CENARIOS = {
    "Conservador": {
        "pct_max_parcela_renda": {
            "A - Baixo risco":       0.35,
            "B - Médio-baixo risco": 0.30,
            "C - Médio risco":       0.25,
            "D - Alto risco":        None,
            "E - Muito alto risco":  None,
        },
        "acao_faixa_d": "Análise manual",
        "acao_faixa_e": "Recusar",
        "redutor_restritivo": {
            "sem_restritivo": 1.00,
            "ate_2pct":       0.90,
            "ate_5pct":       0.75,
            "ate_10pct":      0.60,
            "acima_10pct":    0.40,
        },
        "redutor_inativo": 0.80,
        "redutor_tempo_conta": {
            "curto": 0.85,
            "medio": 0.95,
            "longo": 1.00,
        },
        "valor_minimo_aprovacao": 500.0,
        "restritivos_aprovacao_d": [],
    },
    "Equilibrado": {
        "pct_max_parcela_renda": {
            "A - Baixo risco":       0.40,
            "B - Médio-baixo risco": 0.35,
            "C - Médio risco":       0.30,
            "D - Alto risco":        None,
            "E - Muito alto risco":  None,
        },
        "acao_faixa_d": "Análise manual",
        "acao_faixa_e": "Recusar",
        "redutor_restritivo": {
            "sem_restritivo": 1.00,
            "ate_2pct":       0.92,
            "ate_5pct":       0.80,
            "ate_10pct":      0.65,
            "acima_10pct":    0.45,
        },
        "redutor_inativo": 0.85,
        "redutor_tempo_conta": {
            "curto": 0.88,
            "medio": 0.97,
            "longo": 1.00,
        },
        "valor_minimo_aprovacao": 500.0,
        "restritivos_aprovacao_d": [],
    },
    "Expansivo": {
        "pct_max_parcela_renda": {
            "A - Baixo risco":       0.45,
            "B - Médio-baixo risco": 0.40,
            "C - Médio risco":       0.35,
            "D - Alto risco":        0.25,
            "E - Muito alto risco":  None,
        },
        "acao_faixa_d": "condicional",
        "acao_faixa_e": "Recusar",
        "redutor_restritivo": {
            "sem_restritivo": 1.00,
            "ate_2pct":       0.95,
            "ate_5pct":       0.85,
            "ate_10pct":      0.70,
            "acima_10pct":    0.50,
        },
        "redutor_inativo": 0.90,
        "redutor_tempo_conta": {
            "curto": 0.92,
            "medio": 0.98,
            "longo": 1.00,
        },
        "valor_minimo_aprovacao": 500.0,
        # Faixa D aprovada com reducao apenas se restritivos <= 5% da renda
        "restritivos_aprovacao_d": ["sem_restritivo", "ate_2pct", "ate_5pct"],
    },
}

print("Cenarios definidos com sucesso.")
for nome in CENARIOS:
    print(f"  - {nome}")

Cenarios definidos com sucesso.
  - Conservador
  - Equilibrado
  - Expansivo


In [9]:
# Salvar parametros para auditoria
registros_params = []
for nome_c, params in CENARIOS.items():
    for faixa, pct in params["pct_max_parcela_renda"].items():
        registros_params.append({
            "cenario":                   nome_c,
            "faixa_risco":              faixa,
            "pct_max_parcela_renda":    pct,
            "acao_faixa_d":             params["acao_faixa_d"],
            "acao_faixa_e":             params["acao_faixa_e"],
            "redutor_inativo":          params["redutor_inativo"],
            "redutor_rest_sem":         params["redutor_restritivo"]["sem_restritivo"],
            "redutor_rest_ate2pct":     params["redutor_restritivo"]["ate_2pct"],
            "redutor_rest_ate5pct":     params["redutor_restritivo"]["ate_5pct"],
            "redutor_rest_ate10pct":    params["redutor_restritivo"]["ate_10pct"],
            "redutor_rest_acima10pct":  params["redutor_restritivo"]["acima_10pct"],
            "redutor_tempo_curto":      params["redutor_tempo_conta"]["curto"],
            "redutor_tempo_medio":      params["redutor_tempo_conta"]["medio"],
            "redutor_tempo_longo":      params["redutor_tempo_conta"]["longo"],
            "valor_minimo_aprovacao":   params["valor_minimo_aprovacao"],
        })

df_params = pd.DataFrame(registros_params)
df_params.to_csv(TABLES_DIR / "politica_parametros_cenarios.csv", index=False)
print(f"Parametros salvos -> {TABLES_DIR / 'politica_parametros_cenarios.csv'}")
df_params

Parametros salvos -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_parametros_cenarios.csv


,cenario,faixa_risco,pct_max_parcela_renda,acao_faixa_d,acao_faixa_e,redutor_inativo,redutor_rest_sem,redutor_rest_ate2pct,redutor_rest_ate5pct,redutor_rest_ate10pct,redutor_rest_acima10pct,redutor_tempo_curto,redutor_tempo_medio,redutor_tempo_longo,valor_minimo_aprovacao
0,Conservador,A - Baixo risco,0.3500,Análise manual,Recusar,0.8000,1.0000,0.9000,0.7500,0.6000,0.4000,0.8500,0.9500,1.0000,500.0000
1,Conservador,B - Médio-baixo risco,0.3000,Análise manual,Recusar,0.8000,1.0000,0.9000,0.7500,0.6000,0.4000,0.8500,0.9500,1.0000,500.0000
2,Conservador,C - Médio risco,0.2500,Análise manual,Recusar,0.8000,1.0000,0.9000,0.7500,0.6000,0.4000,0.8500,0.9500,1.0000,500.0000
3,Conservador,D - Alto risco,NaN,Análise manual,Recusar,0.8000,1.0000,0.9000,0.7500,0.6000,0.4000,0.8500,0.9500,1.0000,500.0000
4,Conservador,E - Muito alto risco,NaN,Análise manual,Recusar,0.8000,1.0000,0.9000,0.7500,0.6000,0.4000,0.8500,0.9500,1.0000,500.0000
5,Equilibrado,A - Baixo risco,0.4000,Análise manual,Recusar,0.8500,1.0000,0.9200,0.8000,0.6500,0.4500,0.8800,0.9700,1.0000,500.0000
6,Equilibrado,B - Médio-baixo risco,0.3500,Análise manual,Recusar,0.8500,1.0000,0.9200,0.8000,0.6500,0.4500,0.8800,0.9700,1.0000,500.0000
7,Equilibrado,C - Médio risco,0.3000,Análise manual,Recusar,0.8500,1.0000,0.9200,0.8000,0.6500,0.4500,0.8800,0.9700,1.0000,500.0000
8,Equilibrado,D - Alto risco,NaN,Análise manual,Recusar,0.8500,1.0000,0.9200,0.8000,0.6500,0.4500,0.8800,0.9700,1.0000,500.0000
9,Equilibrado,E - Muito alto risco,NaN,Análise manual,Recusar,0.8500,1.0000,0.9200,0.8000,0.6500,0.4500,0.8800,0.9700,1.0000,500.0000


## 7. Funções de simulação da política

In [10]:
def classificar_restritivo_sobre_renda(valor):
    """Classifica nivel de restritivos em relacao a renda."""
    if pd.isna(valor) or valor <= 0:
        return "sem_restritivo"
    elif valor <= 0.02:
        return "ate_2pct"
    elif valor <= 0.05:
        return "ate_5pct"
    elif valor <= 0.10:
        return "ate_10pct"
    else:
        return "acima_10pct"


def classificar_tempo_relacionamento(tempo_anos):
    """Classifica tempo de relacionamento com o banco."""
    if pd.isna(tempo_anos) or tempo_anos < 1:
        return "curto"
    elif tempo_anos < 3:
        return "medio"
    else:
        return "longo"


def calcular_valor_presente_parcelas(taxa, prazo):
    """Fator VP de anuidade: (1 - (1+taxa)^(-prazo)) / taxa."""
    taxa  = taxa  if (not pd.isna(taxa)  and taxa  > 0) else 0.0001
    prazo = prazo if (not pd.isna(prazo) and prazo > 0) else 1
    return (1 - (1 + taxa) ** (-prazo)) / taxa


def simular_cenario_politica(df_entrada, nome_cenario, params):
    """Aplica a politica de concessao para um cenario."""
    res = df_entrada.copy()
    res["cenario"] = nome_cenario

    # Classificacoes auxiliares
    res["classe_restritivo_c"] = res["restritivos_sobre_renda"].apply(
        classificar_restritivo_sobre_renda
    )
    res["classe_tempo_conta_c"] = res["tempo_conta_anos"].apply(
        classificar_tempo_relacionamento
    )

    # Redutores
    res["redutor_restritivo_c"] = res["classe_restritivo_c"].map(
        params["redutor_restritivo"]
    )
    res["redutor_tempo_c"] = res["classe_tempo_conta_c"].map(
        params["redutor_tempo_conta"]
    )
    red_inativo = params["redutor_inativo"]
    res["redutor_inativo_c"] = res["flag_cliente_ativo"].apply(
        lambda x: 1.0 if x == 1 else red_inativo
    )

    # Percentual maximo de parcela/renda por faixa
    pct_map = params["pct_max_parcela_renda"]
    res["pct_max_c"] = res["faixa_risco"].map(pct_map)

    # Parcela maxima
    def _parcela_max(row):
        renda = row["valor_renda"]
        pct   = row["pct_max_c"]
        if pd.isna(renda) or renda <= 0 or pct is None or pd.isna(pct):
            return 0.0
        return (
            renda
            * pct
            * row["redutor_restritivo_c"]
            * row["redutor_inativo_c"]
            * row["redutor_tempo_c"]
        )

    res["parcela_maxima_c"] = res.apply(_parcela_max, axis=1)

    # Valor maximo sugerido (formula VP)
    res["valor_maximo_c"] = res.apply(
        lambda r: max(
            0.0,
            r["parcela_maxima_c"] * calcular_valor_presente_parcelas(
                r["valor_taxa"], r["valor_prazo"]
            ),
        ),
        axis=1,
    )

    # Decisao
    valor_min      = params["valor_minimo_aprovacao"]
    acao_d         = params["acao_faixa_d"]
    restritivos_d  = set(params.get("restritivos_aprovacao_d", []))

    def _decisao(row):
        faixa = row["faixa_risco"]

        if faixa == "E - Muito alto risco":
            return "Recusar"

        if faixa == "D - Alto risco":
            if acao_d == "Análise manual":
                return "Análise manual"
            elif acao_d == "condicional":
                if (
                    row["classe_restritivo_c"] in restritivos_d
                    and row["valor_maximo_c"] >= valor_min
                ):
                    return "Aprovar valor reduzido"
                else:
                    return "Análise manual"
            else:
                return "Análise manual"

        # Faixas A, B, C
        vm = row["valor_maximo_c"]
        ve = row["valor_emprestado"]
        if vm >= ve:
            return "Aprovar valor solicitado"
        elif vm >= valor_min:
            return "Aprovar valor reduzido"
        else:
            return "Recusar"

    res["decisao_cenario"] = res.apply(_decisao, axis=1)

    # Valor aprovado automatico
    def _valor_aprov(row):
        d = row["decisao_cenario"]
        if d == "Aprovar valor solicitado":
            return row["valor_emprestado"]
        elif d == "Aprovar valor reduzido":
            return min(row["valor_maximo_c"], row["valor_emprestado"])
        else:
            return 0.0

    res["valor_aprovado_c"] = res.apply(_valor_aprov, axis=1)

    return res


def calcular_resumo_cenario(df_cenario):
    """Calcula indicadores executivos de um cenario."""
    n = len(df_cenario)
    mask_aprov = df_cenario["decisao_cenario"].isin(
        ["Aprovar valor solicitado", "Aprovar valor reduzido"]
    )
    df_aprov = df_cenario[mask_aprov]
    val_orig  = df_cenario["valor_emprestado"].sum()
    val_aprov = df_cenario["valor_aprovado_c"].sum()
    pct_exp   = val_aprov / val_orig if val_orig > 0 else 0.0

    return {
        "cenario":                       df_cenario["cenario"].iloc[0],
        "qtd_operacoes":                 n,
        "taxa_historica_inadimplencia":  df_cenario["target_inadimplente_12m"].mean(),
        "taxa_aprovacao_automatica":     mask_aprov.mean(),
        "taxa_analise_manual":           (df_cenario["decisao_cenario"] == "Análise manual").mean(),
        "taxa_recusa":                   (df_cenario["decisao_cenario"] == "Recusar").mean(),
        "inadimplencia_aprovados":       df_aprov["target_inadimplente_12m"].mean() if len(df_aprov) > 0 else np.nan,
        "pd_media_aprovados":            df_aprov["pd_score"].mean() if len(df_aprov) > 0 else np.nan,
        "valor_original_total":          val_orig,
        "valor_aprovado_total":          val_aprov,
        "pct_exposicao_aprovada":        pct_exp,
        "reducao_exposicao":             1.0 - pct_exp,
        "valor_medio_aprovado":          df_aprov["valor_aprovado_c"].mean() if len(df_aprov) > 0 else 0.0,
    }


def resumir_decisoes_por_cenario(df_cenario):
    """Agrupa metricas por decisao dentro de um cenario."""
    nome    = df_cenario["cenario"].iloc[0]
    n_total = len(df_cenario)
    registros = []
    for decisao, grupo in df_cenario.groupby("decisao_cenario", observed=True):
        n = len(grupo)
        registros.append({
            "cenario":               nome,
            "decisao":              decisao,
            "qtd_operacoes":        n,
            "participacao_operacoes": n / n_total,
            "bad_rate_observado":   grupo["target_inadimplente_12m"].mean(),
            "pd_media":             grupo["pd_score"].mean(),
            "valor_original_total": grupo["valor_emprestado"].sum(),
            "valor_aprovado_total": grupo["valor_aprovado_c"].sum(),
            "ticket_medio_original":grupo["valor_emprestado"].mean(),
            "ticket_medio_aprovado":grupo["valor_aprovado_c"].mean(),
        })
    return pd.DataFrame(registros)


def resumir_faixas_por_cenario(df_cenario):
    """Agrupa metricas por faixa de risco dentro de um cenario."""
    nome = df_cenario["cenario"].iloc[0]
    ordem_f = [
        "A - Baixo risco", "B - Médio-baixo risco",
        "C - Médio risco", "D - Alto risco", "E - Muito alto risco",
    ]
    registros = []
    for faixa in ordem_f:
        grupo = df_cenario[df_cenario["faixa_risco"] == faixa]
        if len(grupo) == 0:
            continue
        mask_aprov = grupo["decisao_cenario"].isin(
            ["Aprovar valor solicitado", "Aprovar valor reduzido"]
        )
        val_orig  = grupo["valor_emprestado"].sum()
        val_aprov = grupo["valor_aprovado_c"].sum()
        registros.append({
            "cenario":               nome,
            "faixa_risco":           faixa,
            "qtd_operacoes":         len(grupo),
            "bad_rate_observado":    grupo["target_inadimplente_12m"].mean(),
            "pd_media":              grupo["pd_score"].mean(),
            "taxa_aprovacao":        mask_aprov.mean(),
            "taxa_analise_manual":   (grupo["decisao_cenario"] == "Análise manual").mean(),
            "taxa_recusa":           (grupo["decisao_cenario"] == "Recusar").mean(),
            "valor_original_total":  val_orig,
            "valor_aprovado_total":  val_aprov,
            "pct_exposicao_aprovada": val_aprov / val_orig if val_orig > 0 else 0.0,
        })
    return pd.DataFrame(registros)


print("Funcoes de simulacao definidas com sucesso.")

Funcoes de simulacao definidas com sucesso.


## 8. Simulação dos cenários

In [11]:
resultados = {}
for nome, params in CENARIOS.items():
    df_res = simular_cenario_politica(df, nome, params)
    resultados[nome] = df_res
    r = calcular_resumo_cenario(df_res)
    print(
        f"[{nome:12s}] "
        f"Aprovacao: {r['taxa_aprovacao_automatica']:.1%} | "
        f"Manual: {r['taxa_analise_manual']:.1%} | "
        f"Recusa: {r['taxa_recusa']:.1%} | "
        f"Bad rate aprov.: {r['inadimplencia_aprovados']:.2%}"
    )

# Concatenar base completa de cenarios
df_cenarios = pd.concat(list(resultados.values()), ignore_index=True)
print(f"\nBase de cenarios: {df_cenarios.shape[0]:,} linhas x {df_cenarios.shape[1]} colunas")

caminho_cenarios = PROCESSED_DIR / "base_simulacao_cenarios_politica.parquet"
df_cenarios.to_parquet(caminho_cenarios, index=False)
print(f"Base salva -> {caminho_cenarios}")

[Conservador ] Aprovacao: 74.9% | Manual: 15.0% | Recusa: 10.1% | Bad rate aprov.: 3.90%
[Equilibrado ] Aprovacao: 75.0% | Manual: 15.0% | Recusa: 10.0% | Bad rate aprov.: 3.90%
[Expansivo   ] Aprovacao: 86.7% | Manual: 3.3% | Recusa: 10.0% | Bad rate aprov.: 6.74%

Base de cenarios: 14,586 linhas x 52 colunas
Base salva -> C:\GitHub\datascience\projetos\concessao_credito\data\processed\base_simulacao_cenarios_politica.parquet


In [12]:
# Resumo executivo
resumos = [calcular_resumo_cenario(r) for r in resultados.values()]
df_resumo = pd.DataFrame(resumos)
df_resumo.to_csv(TABLES_DIR / "politica_resumo_cenarios.csv", index=False)
print(f"Resumo salvo -> {TABLES_DIR / 'politica_resumo_cenarios.csv'}")
df_resumo

Resumo salvo -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_resumo_cenarios.csv


,cenario,qtd_operacoes,taxa_historica_inadimplencia,taxa_aprovacao_automatica,taxa_analise_manual,taxa_recusa,inadimplencia_aprovados,pd_media_aprovados,valor_original_total,valor_aprovado_total,pct_exposicao_aprovada,reducao_exposicao,valor_medio_aprovado
0,Conservador,4862,0.1230,0.7489,0.1499,0.1012,0.0390,0.0501,"64,604,367.2200","23,494,528.8043",0.3637,0.6363,"6,452.7681"
1,Equilibrado,4862,0.1230,0.7497,0.1499,0.1004,0.0390,0.0501,"64,604,367.2200","27,239,611.0546",0.4216,0.5784,"7,473.1443"
2,Expansivo,4862,0.1230,0.8665,0.0333,0.1002,0.0674,0.0734,"64,604,367.2200","34,192,017.5838",0.5293,0.4707,"8,115.8361"


In [13]:
# Resumo por decisao
df_decisoes = pd.concat(
    [resumir_decisoes_por_cenario(r) for r in resultados.values()], ignore_index=True
)
df_decisoes.to_csv(TABLES_DIR / "politica_resumo_decisoes_cenarios.csv", index=False)
print(f"Decisoes salvas -> {TABLES_DIR / 'politica_resumo_decisoes_cenarios.csv'}")
df_decisoes

Decisoes salvas -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_resumo_decisoes_cenarios.csv


,cenario,decisao,qtd_operacoes,participacao_operacoes,bad_rate_observado,pd_media,valor_original_total,valor_aprovado_total,ticket_medio_original,ticket_medio_aprovado
0,Conservador,Análise manual,729,0.1499,0.2510,0.2253,"10,765,895.7700",0.0000,"14,768.0326",0.0000
1,Conservador,Aprovar valor reduzido,2651,0.5452,0.0502,0.0610,"41,107,723.8500","18,202,640.0543","15,506.4971","6,866.3297"
2,Conservador,Aprovar valor solicitado,990,0.2036,0.0091,0.0208,"5,291,888.7500","5,291,888.7500","5,345.3422","5,345.3422"
3,Conservador,Recusar,492,0.1012,0.5549,0.5181,"7,438,858.8500",0.0000,"15,119.6318",0.0000
4,Equilibrado,Análise manual,729,0.1499,0.2510,0.2253,"10,765,895.7700",0.0000,"14,768.0326",0.0000
5,Equilibrado,Aprovar valor reduzido,2411,0.4959,0.0523,0.0633,"39,460,495.6900","20,294,250.1846","16,366.8584","8,417.3580"
6,Equilibrado,Aprovar valor solicitado,1234,0.2538,0.0130,0.0244,"6,945,360.8700","6,945,360.8700","5,628.3313","5,628.3313"
7,Equilibrado,Recusar,488,0.1004,0.5594,0.5216,"7,432,614.8900",0.0000,"15,230.7682",0.0000
8,Expansivo,Análise manual,162,0.0333,0.2531,0.2336,"1,451,312.0700",0.0000,"8,958.7165",0.0000
9,Expansivo,Aprovar valor reduzido,2718,0.5590,0.0946,0.0980,"46,566,740.3000","25,037,414.3238","17,132.7227","9,211.7050"


In [14]:
# Resumo por faixa de risco
df_faixas = pd.concat(
    [resumir_faixas_por_cenario(r) for r in resultados.values()], ignore_index=True
)
df_faixas.to_csv(TABLES_DIR / "politica_resumo_faixas_cenarios.csv", index=False)
print(f"Faixas salvas -> {TABLES_DIR / 'politica_resumo_faixas_cenarios.csv'}")
df_faixas

Faixas salvas -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_resumo_faixas_cenarios.csv


,cenario,faixa_risco,qtd_operacoes,bad_rate_observado,pd_media,taxa_aprovacao,taxa_analise_manual,taxa_recusa,valor_original_total,valor_aprovado_total,pct_exposicao_aprovada
0,Conservador,A - Baixo risco,1945,0.0108,0.0225,0.9995,0.0000,0.0005,"21,279,049.3200","13,535,388.7266",0.6361
1,Conservador,B - Médio-baixo risco,972,0.0453,0.0607,1.0000,0.0000,0.0000,"14,219,232.3700","6,261,825.6050",0.4404
2,Conservador,C - Médio risco,729,0.1056,0.1098,0.9945,0.0000,0.0055,"10,908,478.1700","3,697,314.4726",0.3389
3,Conservador,D - Alto risco,729,0.2510,0.2253,0.0000,1.0000,0.0000,"10,765,895.7700",0.0000,0.0000
4,Conservador,E - Muito alto risco,487,0.5606,0.5224,0.0000,0.0000,1.0000,"7,431,711.5900",0.0000,0.0000
5,Equilibrado,A - Baixo risco,1945,0.0108,0.0225,1.0000,0.0000,0.0000,"21,279,049.3200","14,971,939.0918",0.7036
6,Equilibrado,B - Médio-baixo risco,972,0.0453,0.0607,1.0000,0.0000,0.0000,"14,219,232.3700","7,561,101.0019",0.5318
7,Equilibrado,C - Médio risco,729,0.1056,0.1098,0.9986,0.0000,0.0014,"10,908,478.1700","4,706,570.9608",0.4315
8,Equilibrado,D - Alto risco,729,0.2510,0.2253,0.0000,1.0000,0.0000,"10,765,895.7700",0.0000,0.0000
9,Equilibrado,E - Muito alto risco,487,0.5606,0.5224,0.0000,0.0000,1.0000,"7,431,711.5900",0.0000,0.0000


In [15]:
# Score por decisão e faixa de risco por cenário
df_score_dec = (
    df_cenarios
    .groupby(["cenario", "faixa_risco", "decisao_cenario"], observed=True)
    .agg(
        qtd_operacoes=("pd_score", "count"),
        pd_medio=("pd_score", "mean"),
        bad_rate=("target_inadimplente_12m", "mean"),
        valor_original=("valor_emprestado", "sum"),
        valor_aprovado=("valor_aprovado_c", "sum"),
    )
    .reset_index()
)
df_score_dec.to_csv(
    TABLES_DIR / "politica_score_por_decisao_cenarios.csv",
    index=False,
)
print(f"Score por decisão salvo -> {TABLES_DIR / 'politica_score_por_decisao_cenarios.csv'}")
df_score_dec.head(20)

Score por decisão salvo -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_score_por_decisao_cenarios.csv


,cenario,faixa_risco,decisao_cenario,qtd_operacoes,pd_medio,bad_rate,valor_original,valor_aprovado
0,Conservador,A - Baixo risco,Aprovar valor reduzido,1043,0.0273,0.0163,"16,355,156.1400","8,612,109.5166"
1,Conservador,A - Baixo risco,Aprovar valor solicitado,901,0.0169,0.0044,"4,923,279.2100","4,923,279.2100"
2,Conservador,A - Baixo risco,Recusar,1,0.0165,0.0000,613.9700,0.0000
3,Conservador,B - Médio-baixo risco,Aprovar valor reduzido,890,0.0610,0.0461,"13,876,528.7600","5,919,121.9950"
4,Conservador,B - Médio-baixo risco,Aprovar valor solicitado,82,0.0570,0.0366,"342,703.6100","342,703.6100"
5,Conservador,C - Médio risco,Aprovar valor reduzido,718,0.1098,0.1045,"10,876,038.9500","3,671,408.5426"
6,Conservador,C - Médio risco,Aprovar valor solicitado,7,0.1044,0.2857,"25,905.9300","25,905.9300"
7,Conservador,C - Médio risco,Recusar,4,0.1149,0.0000,"6,533.2900",0.0000
8,Conservador,D - Alto risco,Análise manual,729,0.2253,0.2510,"10,765,895.7700",0.0000
9,Conservador,E - Muito alto risco,Recusar,487,0.5224,0.5606,"7,431,711.5900",0.0000


## 9. Comparação executiva dos cenários

In [16]:
COR_CENARIO = {
    "Conservador": CORES["verde_escuro"],
    "Equilibrado": CORES["verde_vivo"],
    "Expansivo":   CORES["amarelo"],
}

DECISOES_ORDEM = ["Aprovar valor solicitado", "Aprovar valor reduzido", "Análise manual", "Recusar"]
COR_DECISAO = {
    "Aprovar valor solicitado": CORES["verde_principal"],
    "Aprovar valor reduzido":   CORES["verde_vivo"],
    "Análise manual":           CORES["amarelo"],
    "Recusar":                  CORES["marrom"],
}

print("Cores definidas.")

Cores definidas.


In [17]:
# Grafico 1 — Trade-off: aprovacao x inadimplencia dos aprovados
df_p1 = df_resumo.copy()
df_p1["aprovacao_pct"]    = df_p1["taxa_aprovacao_automatica"] * 100
df_p1["inadimplencia_pct"]= df_p1["inadimplencia_aprovados"] * 100
df_p1["exposicao_mm"]     = df_p1["valor_aprovado_total"] / 1e6

fig1 = go.Figure()
for _, row in df_p1.iterrows():
    fig1.add_trace(
        go.Scatter(
            x=[row["aprovacao_pct"]],
            y=[row["inadimplencia_pct"]],
            mode="markers+text",
            marker=dict(
                size=row["exposicao_mm"] * 1.6,
                color=COR_CENARIO[row["cenario"]],
                opacity=0.88,
                line=dict(width=1.5, color="white"),
            ),
            text=[row["cenario"]],
            textposition="top center",
            textfont=dict(size=13, color=CORES["cinza_texto"]),
            name=row["cenario"],
            hovertemplate=(
                f"<b>{row['cenario']}</b><br>"
                f"Aprovacao automatica: {row['aprovacao_pct']:.1f}%<br>"
                f"Inadimplencia aprovados: {row['inadimplencia_pct']:.2f}%<br>"
                f"Exposicao aprovada: R$ {row['exposicao_mm']:.1f} MM"
                "<extra></extra>"
            ),
        )
    )

fig1.add_hline(
    y=TAXA_HISTORICA * 100,
    line_dash="dash", line_color=CORES["cinza_medio"],
    annotation_text=f"Taxa historica ({TAXA_HISTORICA:.1%})",
    annotation_position="right",
)

fig1 = aplicar_layout(
    fig1,
    titulo="Trade-off entre aprovação automática e inadimplência dos aprovados",
    subtitulo="Tamanho do ponto proporcional à exposição aprovada (R$ MM) | Linha = taxa histórica",
    altura=520,
)
fig1.update_xaxes(title_text="Taxa de aprovação automática (%)")
fig1.update_yaxes(title_text="Inadimplência dos aprovados (%)")

# O HTML consolidado do notebook deve ser gerado depois com jupyter nbconvert.
fig1.show()
print("Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.")

Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.


In [18]:
# Grafico 2 — Taxas de decisao por cenario
df_p2 = df_decisoes.copy()
df_p2["participacao_pct"] = df_p2["participacao_operacoes"] * 100

fig2 = go.Figure()
for decisao in DECISOES_ORDEM:
    sub = df_p2[df_p2["decisao"] == decisao]
    fig2.add_trace(
        go.Bar(
            x=sub["cenario"],
            y=sub["participacao_pct"],
            name=decisao,
            marker_color=COR_DECISAO[decisao],
            text=sub["participacao_pct"].apply(lambda v: f"{v:.1f}%"),
            textposition="outside",
            textfont=dict(size=11),
        )
    )

fig2 = aplicar_layout(
    fig2,
    titulo="Distribuição das decisões por cenário",
    subtitulo="Participação percentual por tipo de decisão — base de validação",
    altura=520,
)
fig2.update_layout(barmode="group")
fig2.update_xaxes(title_text="Cenário")
fig2.update_yaxes(title_text="Participação (%)", range=[0, 80])

# O HTML consolidado do notebook deve ser gerado depois com jupyter nbconvert.
fig2.show()
print("Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.")

Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.


In [19]:
# Grafico 3 — Inadimplencia dos aprovados por cenario
df_p3 = df_resumo.copy()
df_p3["inad_pct"] = df_p3["inadimplencia_aprovados"] * 100

fig3 = go.Figure()
fig3.add_trace(
    go.Bar(
        x=df_p3["cenario"],
        y=df_p3["inad_pct"],
        marker_color=[COR_CENARIO[c] for c in df_p3["cenario"]],
        text=df_p3["inad_pct"].apply(lambda v: f"{v:.2f}%"),
        textposition="outside",
        textfont=dict(size=13),
        name="Inadimplência aprovados",
    )
)
fig3.add_hline(
    y=TAXA_HISTORICA * 100,
    line_dash="dash", line_color=CORES["cinza_medio"],
    annotation_text=f"Taxa histórica ({TAXA_HISTORICA:.1%})",
    annotation_position="right",
)
fig3 = aplicar_layout(
    fig3,
    titulo="Inadimplência dos aprovados por cenário",
    subtitulo="Backtest histórico | Linha = taxa histórica total da base",
    altura=480,
)
fig3.update_xaxes(title_text="Cenário")
fig3.update_yaxes(title_text="Inadimplência dos aprovados (%)")

# O HTML consolidado do notebook deve ser gerado depois com jupyter nbconvert.
fig3.show()
print("Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.")

Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.


In [20]:
# Grafico 4 — Exposicao aprovada por cenario
df_p4 = df_resumo.copy()
df_p4["exp_mm"]   = df_p4["valor_aprovado_total"] / 1e6
df_p4["orig_mm"]  = df_p4["valor_original_total"] / 1e6
df_p4["pct_text"] = df_p4["pct_exposicao_aprovada"].apply(lambda v: f"{v:.1%}")

fig4 = go.Figure()
fig4.add_trace(
    go.Bar(
        x=df_p4["cenario"],
        y=df_p4["exp_mm"],
        marker_color=[COR_CENARIO[c] for c in df_p4["cenario"]],
        text=df_p4.apply(lambda r: f"R$ {r['exp_mm']:.1f} MM ({r['pct_text']})", axis=1),
        textposition="outside",
        textfont=dict(size=11),
        name="Exposição aprovada",
    )
)
orig_mm = df_p4["orig_mm"].iloc[0]
fig4.add_hline(
    y=orig_mm,
    line_dash="dash", line_color=CORES["cinza_medio"],
    annotation_text=f"Exposição original total (R$ {orig_mm:.1f} MM)",
    annotation_position="right",
)
fig4 = aplicar_layout(
    fig4,
    titulo="Exposição financeira aprovada por cenário",
    subtitulo="Valor automaticamente aprovado em R$ MM | Percentual sobre exposição original",
    altura=480,
)
fig4.update_xaxes(title_text="Cenário")
fig4.update_yaxes(title_text="Exposição aprovada (R$ MM)")

# O HTML consolidado do notebook deve ser gerado depois com jupyter nbconvert.
fig4.show()
print("Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.")

Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.


In [21]:
# Grafico 5 — Distribuicao de decisoes por faixa de risco (subplots por cenario)
ORDEM_FAIXAS = [
    "A - Baixo risco", "B - Médio-baixo risco",
    "C - Médio risco", "D - Alto risco", "E - Muito alto risco",
]
LABELS_FAIXAS = ["A", "B", "C", "D", "E"]

nomes_cenarios = list(CENARIOS.keys())
fig5 = make_subplots(
    rows=1, cols=len(nomes_cenarios),
    subplot_titles=nomes_cenarios,
    shared_yaxes=True,
)

for col_i, nome_c in enumerate(nomes_cenarios, start=1):
    df_c = resultados[nome_c]
    for decisao in DECISOES_ORDEM:
        y_vals, x_labels = [], []
        for faixa, label in zip(ORDEM_FAIXAS, LABELS_FAIXAS):
            grupo = df_c[df_c["faixa_risco"] == faixa]
            if len(grupo) == 0:
                y_vals.append(0)
            else:
                y_vals.append((grupo["decisao_cenario"] == decisao).mean() * 100)
            x_labels.append(label)
        fig5.add_trace(
            go.Bar(
                x=x_labels,
                y=y_vals,
                name=decisao,
                marker_color=COR_DECISAO[decisao],
                showlegend=(col_i == 1),
                text=[f"{v:.0f}%" if v >= 5 else "" for v in y_vals],
                textposition="inside",
                textfont=dict(size=9, color="white"),
            ),
            row=1, col=col_i,
        )

fig5.update_layout(
    barmode="stack",
    height=520,
    title={
        "text": "Distribuição das decisões por faixa de risco<br>"
                "<sup>Cada barra = 100% das operações naquela faixa e cenário</sup>",
        "x": 0.02, "xanchor": "left",
        "font": {"size": 18, "color": CORES["cinza_texto"]},
    },
    font=dict(family="Arial", color=CORES["cinza_texto"], size=12),
    paper_bgcolor=CORES["branco"],
    plot_bgcolor=CORES["branco"],
    legend=dict(orientation="h", y=-0.18, x=0.5, xanchor="center"),
    margin=dict(l=60, r=40, t=100, b=110),
)
fig5.update_yaxes(title_text="Participação (%)", col=1, range=[0, 105])
fig5.update_xaxes(title_text="Faixa de risco")

# O HTML consolidado do notebook deve ser gerado depois com jupyter nbconvert.
fig5.show()
print("Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.")

Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.


In [22]:
# Grafico 6 — Valor original vs valor aprovado por cenario
df_p6 = df_resumo.copy()
df_p6["val_orig_mm"] = df_p6["valor_original_total"] / 1e6
df_p6["val_aprov_mm"]= df_p6["valor_aprovado_total"] / 1e6

fig6 = go.Figure()
fig6.add_trace(
    go.Bar(
        x=df_p6["cenario"],
        y=df_p6["val_orig_mm"],
        name="Valor original solicitado",
        marker_color=CORES["cinza_medio"],
        opacity=0.5,
        text=df_p6["val_orig_mm"].apply(lambda v: f"R$ {v:.1f} MM"),
        textposition="outside",
        textfont=dict(size=11),
    )
)
fig6.add_trace(
    go.Bar(
        x=df_p6["cenario"],
        y=df_p6["val_aprov_mm"],
        name="Valor aprovado automaticamente",
        marker_color=CORES["verde_principal"],
        text=df_p6.apply(
            lambda r: f"R$ {r['val_aprov_mm']:.1f} MM ({r['pct_exposicao_aprovada']:.1%})", axis=1
        ),
        textposition="outside",
        textfont=dict(size=11),
    )
)
fig6 = aplicar_layout(
    fig6,
    titulo="Valor original solicitado versus valor aprovado automaticamente",
    subtitulo="R$ milhões | Percentual = proporção aprovada sobre o total solicitado",
    altura=500,
)
fig6.update_layout(barmode="group")
fig6.update_xaxes(title_text="Cenário")
fig6.update_yaxes(title_text="Valor (R$ MM)")

# O HTML consolidado do notebook deve ser gerado depois com jupyter nbconvert.
fig6.show()
print("Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.")

Figura exibida no notebook; o HTML consolidado ser? gerado via nbconvert.


## 10. Escolha do cenário recomendado

A escolha é baseada em um **score gerencial ponderado** com quatro dimensões:

| Dimensão | Peso | Direção |
|---|---|---|
| Controle de inadimplência | 40% | Menor inadimplência dos aprovados = melhor |
| Preservação da aprovação | 30% | Maior taxa de aprovação automática = melhor |
| Preservação da exposição | 20% | Maior exposição aprovada = melhor |
| Eficiência operacional | 10% | Menor taxa de análise manual = melhor |

Cada métrica é normalizada em [0, 1] antes da ponderação.

In [23]:
def norm_menor_melhor(serie):
    mn, mx = serie.min(), serie.max()
    if mx == mn:
        return pd.Series([1.0] * len(serie), index=serie.index)
    return (mx - serie) / (mx - mn)


def norm_maior_melhor(serie):
    mn, mx = serie.min(), serie.max()
    if mx == mn:
        return pd.Series([1.0] * len(serie), index=serie.index)
    return (serie - mn) / (mx - mn)


NOME_GERENCIAL_CENARIO = {
    "Conservador": "Conservador",
    "Equilibrado": "Equilibrado",
    "Expansivo": "Expansivo com controle de risco",
}


df_score = df_resumo.copy()
df_score["score_inadimplencia"] = norm_menor_melhor(df_score["inadimplencia_aprovados"])
df_score["score_aprovacao"] = norm_maior_melhor(df_score["taxa_aprovacao_automatica"])
df_score["score_exposicao"] = norm_maior_melhor(df_score["pct_exposicao_aprovada"])
df_score["score_eficiencia"] = norm_menor_melhor(df_score["taxa_analise_manual"])

df_score["score_gerencial"] = (
    0.40 * df_score["score_inadimplencia"]
    + 0.30 * df_score["score_aprovacao"]
    + 0.20 * df_score["score_exposicao"]
    + 0.10 * df_score["score_eficiencia"]
)

df_score["nome_gerencial"] = df_score["cenario"].map(NOME_GERENCIAL_CENARIO)

cols_exib = [
    "cenario",
    "taxa_aprovacao_automatica", "inadimplencia_aprovados",
    "pct_exposicao_aprovada", "taxa_analise_manual",
    "score_inadimplencia", "score_aprovacao",
    "score_exposicao", "score_eficiencia",
    "score_gerencial",
]
df_exib = df_score[cols_exib].copy()
for c in ["taxa_aprovacao_automatica", "inadimplencia_aprovados", "pct_exposicao_aprovada", "taxa_analise_manual"]:
    df_exib[c] = df_exib[c].apply(lambda v: f"{v:.2%}")
for c in ["score_inadimplencia", "score_aprovacao", "score_exposicao", "score_eficiencia", "score_gerencial"]:
    df_exib[c] = df_exib[c].apply(lambda v: f"{v:.3f}")

df_exib.to_csv(
    TABLES_DIR / "politica_score_gerencial_cenarios.csv",
    index=False,
)
print(f"Score gerencial salvo -> {TABLES_DIR / 'politica_score_gerencial_cenarios.csv'}")
df_exib

Score gerencial salvo -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_score_gerencial_cenarios.csv


,cenario,taxa_aprovacao_automatica,inadimplencia_aprovados,pct_exposicao_aprovada,taxa_analise_manual,score_inadimplencia,score_aprovacao,score_exposicao,score_eficiencia,score_gerencial
0,Conservador,74.89%,3.90%,36.37%,14.99%,0.998,0.000,0.000,0.000,0.399
1,Equilibrado,74.97%,3.90%,42.16%,14.99%,1.000,0.007,0.350,0.000,0.472
2,Expansivo,86.65%,6.74%,52.93%,3.33%,0.000,1.000,1.000,1.000,0.600


In [24]:
CENARIO_RECOMENDADO = df_score.loc[
    df_score["score_gerencial"].idxmax(), "cenario"
]
NOME_GERENCIAL_RECOMENDADO = NOME_GERENCIAL_CENARIO.get(
    CENARIO_RECOMENDADO, CENARIO_RECOMENDADO
)

row_rec = df_score[df_score["cenario"] == CENARIO_RECOMENDADO].iloc[0]

df_tabela_executiva = pd.DataFrame([{
    "cenario": CENARIO_RECOMENDADO,
    "nome_gerencial": NOME_GERENCIAL_RECOMENDADO,
    "taxa_aprovacao_automatica": row_rec["taxa_aprovacao_automatica"],
    "inadimplencia_aprovados": row_rec["inadimplencia_aprovados"],
    "pct_exposicao_aprovada": row_rec["pct_exposicao_aprovada"],
    "taxa_analise_manual": row_rec["taxa_analise_manual"],
    "taxa_recusa": row_rec["taxa_recusa"],
    "score_gerencial": row_rec["score_gerencial"],
    "recomendacao": (
        f"Recomendar o cenário técnico {CENARIO_RECOMENDADO} como "
        f"{NOME_GERENCIAL_RECOMENDADO}: preserva aprovação e exposição, "
        f"mantendo a inadimplência dos aprovados abaixo da taxa histórica."
    ),
}])

df_tabela_executiva.to_csv(
    TABLES_DIR / "politica_tabela_executiva_recomendacao.csv",
    index=False,
)

print(f"{'='*60}")
print(f"  CENÁRIO RECOMENDADO: {CENARIO_RECOMENDADO.upper()}")
print(f"  NOME GERENCIAL: {NOME_GERENCIAL_RECOMENDADO}")
print(f"{'='*60}")
print(f"  Score gerencial ponderado : {row_rec['score_gerencial']:.3f}")
print(f"  Aprovação automática      : {row_rec['taxa_aprovacao_automatica']:.2%}")
print(f"  Inadimplência aprovados   : {row_rec['inadimplencia_aprovados']:.2%}")
print(f"  Exposição aprovada        : {row_rec['pct_exposicao_aprovada']:.2%}")
print(f"  Análise manual            : {row_rec['taxa_analise_manual']:.2%}")
print(f"  Recusa                    : {row_rec['taxa_recusa']:.2%}")
print(f"{'='*60}")
print(f"Tabela executiva salva -> {TABLES_DIR / 'politica_tabela_executiva_recomendacao.csv'}")

df_tabela_executiva

  CENÁRIO RECOMENDADO: EXPANSIVO
  NOME GERENCIAL: Expansivo com controle de risco
  Score gerencial ponderado : 0.600
  Aprovação automática      : 86.65%
  Inadimplência aprovados   : 6.74%
  Exposição aprovada        : 52.93%
  Análise manual            : 3.33%
  Recusa                    : 10.02%
Tabela executiva salva -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_tabela_executiva_recomendacao.csv


,cenario,nome_gerencial,taxa_aprovacao_automatica,inadimplencia_aprovados,pct_exposicao_aprovada,taxa_analise_manual,taxa_recusa,score_gerencial,recomendacao
0,Expansivo,Expansivo com controle de risco,0.8665,0.0674,0.5293,0.0333,0.1002,0.6000,Recomendar o cenário técnico Expansivo como Ex...


## 11. Política final proposta

O identificador técnico do cenário recomendado permanece como `Expansivo`. Para apresentação gerencial, a recomendação é comunicada como **Expansivo com controle de risco**.

O cenário é recomendado porque preserva mais aprovação automática e exposição financeira que os cenários alternativos, mas ainda mantém a inadimplência dos aprovados em torno de **6,74%**, abaixo da taxa histórica de inadimplência de **12,30%**.

### 11.1 Estrutura consolidada da decisão de crédito

A decisão final combina camadas sequenciais de política:

`PD score -> faixa de risco -> capacidade de pagamento -> redutores -> valor máximo sugerido -> decisão final`

A tabela abaixo explicita o papel de cada camada na política recomendada.

In [25]:
df_estrutura_decisao = pd.DataFrame([
    {"camada": "Rating interno", "papel": "Transformar o pd_score em faixa de risco A-E.", "criterio_politica": "Usa pd_score e faixa_risco; target_inadimplente_12m fica apenas para backtest.", "saida": "Faixa de risco da proposta."},
    {"camada": "Capacidade de pagamento", "papel": "Definir parcela máxima compatível com renda e risco.", "criterio_politica": "Percentual máximo de parcela/renda por faixa no cenário recomendado.", "saida": "Parcela máxima antes de redutores."},
    {"camada": "Restritivos financeiros", "papel": "Reduzir limite quando há pressão financeira relevante.", "criterio_politica": "Redutores por restritivos_sobre_renda: sem, ate_2pct, ate_5pct, ate_10pct e acima_10pct.", "saida": "Parcela máxima ajustada por restritivos."},
    {"camada": "Relacionamento", "papel": "Aplicar governança para cliente inativo e relacionamento curto.", "criterio_politica": "Redutores por flag_cliente_ativo e tempo_conta_anos.", "saida": "Parcela máxima ajustada por relacionamento."},
    {"camada": "Valor máximo sugerido", "papel": "Converter parcela máxima em limite pela taxa e prazo da operação.", "criterio_politica": "Valor presente das parcelas, truncado a zero e limitado ao valor solicitado.", "saida": "valor_maximo_c e valor_aprovado_c."},
    {"camada": "Decisão final", "papel": "Classificar a proposta em aprovação, análise manual ou recusa.", "criterio_politica": "Faixa E recusada; faixa D somente reduzida condicional ou análise manual; A-C conforme capacidade.", "saida": "decisao_cenario."},
    {"camada": "Cenário recomendado", "papel": "Comunicar o apetite de risco escolhido para gestão.", "criterio_politica": f"Identificador técnico: {CENARIO_RECOMENDADO}; nome gerencial: {NOME_GERENCIAL_RECOMENDADO}.", "saida": "Política final recomendada para validação da área de crédito."},
])

df_estrutura_decisao.to_csv(
    TABLES_DIR / "politica_estrutura_consolidada_decisao.csv",
    index=False,
)
print(f"Estrutura consolidada salva -> {TABLES_DIR / 'politica_estrutura_consolidada_decisao.csv'}")
df_estrutura_decisao

Estrutura consolidada salva -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_estrutura_consolidada_decisao.csv


,camada,papel,criterio_politica,saida
0,Rating interno,Transformar o pd_score em faixa de risco A-E.,Usa pd_score e faixa_risco; target_inadimplent...,Faixa de risco da proposta.
1,Capacidade de pagamento,Definir parcela máxima compatível com renda e ...,Percentual máximo de parcela/renda por faixa n...,Parcela máxima antes de redutores.
2,Restritivos financeiros,Reduzir limite quando há pressão financeira re...,"Redutores por restritivos_sobre_renda: sem, at...",Parcela máxima ajustada por restritivos.
3,Relacionamento,Aplicar governança para cliente inativo e rela...,Redutores por flag_cliente_ativo e tempo_conta...,Parcela máxima ajustada por relacionamento.
4,Valor máximo sugerido,Converter parcela máxima em limite pela taxa e...,"Valor presente das parcelas, truncado a zero e...",valor_maximo_c e valor_aprovado_c.
5,Decisão final,"Classificar a proposta em aprovação, análise m...",Faixa E recusada; faixa D somente reduzida con...,decisao_cenario.
6,Cenário recomendado,Comunicar o apetite de risco escolhido para ge...,Identificador técnico: Expansivo; nome gerenci...,Política final recomendada para validação da á...


In [26]:
params_rec = CENARIOS[CENARIO_RECOMENDADO]

intervalos_pd = (
    df
    .groupby("faixa_risco", observed=True)
    .agg(
        pd_min=("pd_score", "min"),
        pd_max=("pd_score", "max"),
    )
    .reset_index()
)

intervalos_pd["intervalo_pd"] = intervalos_pd.apply(
    lambda row: f"{row['pd_min']:.2%} a {row['pd_max']:.2%}",
    axis=1,
)

dict_intervalos_pd = dict(
    zip(intervalos_pd["faixa_risco"], intervalos_pd["intervalo_pd"])
)
dict_pd_min = dict(zip(intervalos_pd["faixa_risco"], intervalos_pd["pd_min"]))
dict_pd_max = dict(zip(intervalos_pd["faixa_risco"], intervalos_pd["pd_max"]))

ACAO_PRINCIPAL = {
    "A - Baixo risco": "Aprovação automática",
    "B - Médio-baixo risco": "Aprovação automática",
    "C - Médio risco": "Aprovação automática",
    "D - Alto risco": "Análise manual" if params_rec["acao_faixa_d"] != "condicional" else "Aprovação reduzida condicional / Análise manual",
    "E - Muito alto risco": "Recusar",
}

OBSERVACAO_NEGOCIO = {
    "A - Baixo risco": "Perfil de menor risco. Aprovar valor solicitado ou reduzido conforme capacidade.",
    "B - Médio-baixo risco": "Risco controlado. Aplicar redutores com moderação.",
    "C - Médio risco": "Risco intermediário. Limite mais restrito e monitoramento por safra.",
    "D - Alto risco": "Risco elevado. Aprovação apenas reduzida e condicional, ou encaminhamento para análise manual.",
    "E - Muito alto risco": "Risco muito alto. Recusa automática na política recomendada.",
}

red_rest = params_rec["redutor_restritivo"]
trat_rest = (
    f"sem={red_rest['sem_restritivo']:.0%} | "
    f"ate2%={red_rest['ate_2pct']:.0%} | "
    f"ate5%={red_rest['ate_5pct']:.0%} | "
    f"ate10%={red_rest['ate_10pct']:.0%} | "
    f"acima10%={red_rest['acima_10pct']:.0%}"
)
red_tc = params_rec["redutor_tempo_conta"]
trat_tempo = (
    f"curto(<1a)={red_tc['curto']:.0%} | "
    f"medio(1-3a)={red_tc['medio']:.0%} | "
    f"longo(>=3a)={red_tc['longo']:.0%}"
)

registros_pol = []
for faixa in ORDEM_FAIXAS:
    pct = params_rec["pct_max_parcela_renda"].get(faixa)
    registros_pol.append({
        "cenario_recomendado": CENARIO_RECOMENDADO,
        "nome_gerencial_cenario": NOME_GERENCIAL_RECOMENDADO,
        "faixa_risco": faixa,
        "pd_min": dict_pd_min[faixa],
        "pd_max": dict_pd_max[faixa],
        "intervalo_pd": dict_intervalos_pd[faixa],
        "acao_principal": ACAO_PRINCIPAL[faixa],
        "pct_max_parcela_renda": f"{pct:.0%}" if pct is not None else "N/A",
        "tratamento_restritivos": trat_rest,
        "tratamento_inativo": f"redutor={params_rec['redutor_inativo']:.0%}",
        "tratamento_tempo_conta": trat_tempo,
        "regra_limite": (
            f"parcela_max = renda x {pct:.0%} x redutores; "
            f"limite = parcela_max x VP(taxa,prazo)" if pct is not None
            else ACAO_PRINCIPAL[faixa]
        ),
        "observacao_negocio": OBSERVACAO_NEGOCIO[faixa],
    })

df_politica_final = pd.DataFrame(registros_pol)
df_politica_final.to_csv(TABLES_DIR / "politica_final_recomendada.csv", index=False)
print(f"Política final salva -> {TABLES_DIR / 'politica_final_recomendada.csv'}")
df_politica_final

Política final salva -> C:\GitHub\datascience\projetos\concessao_credito\outputs\tables\politica_final_recomendada.csv


,cenario_recomendado,nome_gerencial_cenario,faixa_risco,pd_min,pd_max,intervalo_pd,acao_principal,pct_max_parcela_renda,tratamento_restritivos,tratamento_inativo,tratamento_tempo_conta,regra_limite,observacao_negocio
0,Expansivo,Expansivo com controle de risco,A - Baixo risco,0.0047,0.0425,0.47% a 4.25%,Aprovação automática,45%,sem=100% | ate2%=95% | ate5%=85% | ate10%=70% ...,redutor=90%,curto(<1a)=92% | medio(1-3a)=98% | longo(>=3a)...,parcela_max = renda x 45% x redutores; limite ...,Perfil de menor risco. Aprovar valor solicitad...
1,Expansivo,Expansivo com controle de risco,B - Médio-baixo risco,0.0426,0.0823,4.26% a 8.23%,Aprovação automática,40%,sem=100% | ate2%=95% | ate5%=85% | ate10%=70% ...,redutor=90%,curto(<1a)=92% | medio(1-3a)=98% | longo(>=3a)...,parcela_max = renda x 40% x redutores; limite ...,Risco controlado. Aplicar redutores com modera...
2,Expansivo,Expansivo com controle de risco,C - Médio risco,0.0823,0.1465,8.23% a 14.65%,Aprovação automática,35%,sem=100% | ate2%=95% | ate5%=85% | ate10%=70% ...,redutor=90%,curto(<1a)=92% | medio(1-3a)=98% | longo(>=3a)...,parcela_max = renda x 35% x redutores; limite ...,Risco intermediário. Limite mais restrito e mo...
3,Expansivo,Expansivo com controle de risco,D - Alto risco,0.1467,0.3504,14.67% a 35.04%,Aprovação reduzida condicional / Análise manual,25%,sem=100% | ate2%=95% | ate5%=85% | ate10%=70% ...,redutor=90%,curto(<1a)=92% | medio(1-3a)=98% | longo(>=3a)...,parcela_max = renda x 25% x redutores; limite ...,Risco elevado. Aprovação apenas reduzida e con...
4,Expansivo,Expansivo com controle de risco,E - Muito alto risco,0.3505,0.9214,35.05% a 92.14%,Recusar,N/A,sem=100% | ate2%=95% | ate5%=85% | ate10%=70% ...,redutor=90%,curto(<1a)=92% | medio(1-3a)=98% | longo(>=3a)...,Recusar,Risco muito alto. Recusa automática na polític...


In [27]:
# Resumo final impresso e validacoes de governanca das faixas D/E
print(f"{'='*65}")
print(f"  POLITICA FINAL RECOMENDADA: {NOME_GERENCIAL_RECOMENDADO}")
print(f"  Identificador tecnico: {CENARIO_RECOMENDADO}")
print(f"{'='*65}")
print("  Produto: Emprestimo bancario parcelado - Pessoa fisica")
print(f"{'-'*65}")
print("  Formula:")
print("    parcela_max = renda x pct_faixa x red_restritivo x red_inativo x red_tempo")
print("    limite_max  = parcela_max x [(1-(1+taxa)^(-prazo))/taxa]")
print(f"{'-'*65}")
for faixa, pct in params_rec["pct_max_parcela_renda"].items():
    acao = ACAO_PRINCIPAL[faixa]
    val  = f"{pct:.0%}" if pct is not None else f"-> {acao}"
    print(f"  {faixa:28s}: {val}")
print(f"{'-'*65}")
print(f"  Redutor cliente inativo: {params_rec['redutor_inativo']:.0%}")
print(f"  Redutores restritivos (sem/2/5/10/>10%): "
      f"{'/'.join([f'{v:.0%}' for v in params_rec['redutor_restritivo'].values()])}")
print(f"  Redutores tempo conta (curto/medio/longo): "
      f"{'/'.join([f'{v:.0%}' for v in params_rec['redutor_tempo_conta'].values()])}")
print(f"  Valor minimo para aprovacao reduzida: R$ {params_rec['valor_minimo_aprovacao']:,.0f}")
print(f"{'-'*65}")
print("  Resultados historicos (backtest na safra de validacao):")
print(f"    Aprovacao automatica : {row_rec['taxa_aprovacao_automatica']:.2%}")
print(f"    Inadimplencia aprov. : {row_rec['inadimplencia_aprovados']:.2%}")
print(f"    Taxa historica       : {row_rec['taxa_historica_inadimplencia']:.2%}")
print(f"    Exposicao aprovada   : {row_rec['pct_exposicao_aprovada']:.2%} do total")
print(f"    Analise manual       : {row_rec['taxa_analise_manual']:.2%}")
print(f"    Recusa               : {row_rec['taxa_recusa']:.2%}")
print(f"{'-'*65}")

base_rec = resultados[CENARIO_RECOMENDADO]
mask_e = base_rec["faixa_risco"].eq("E - Muito alto risco")
mask_d = base_rec["faixa_risco"].eq("D - Alto risco")
mask_valor_solicitado = base_rec["decisao_cenario"].eq("Aprovar valor solicitado")

qtd_e_nao_recusada = int((mask_e & ~base_rec["decisao_cenario"].eq("Recusar")).sum())
qtd_d_full = int((mask_d & mask_valor_solicitado).sum())
qtd_e_full = int((mask_e & mask_valor_solicitado).sum())

pontos_atencao = []
if qtd_e_nao_recusada > 0:
    pontos_atencao.append(f"Faixa E possui {qtd_e_nao_recusada} propostas nao recusadas.")
if qtd_d_full > 0:
    pontos_atencao.append(f"Faixa D possui {qtd_d_full} aprovacoes de valor solicitado.")
if qtd_e_full > 0:
    pontos_atencao.append(f"Faixa E possui {qtd_e_full} aprovacoes de valor solicitado.")

print("  Validacoes D/E:")
print(f"    Faixa E sempre recusada: {'SIM' if qtd_e_nao_recusada == 0 else 'NAO'}")
print(f"    Faixa D com aprovacao de valor solicitado: {qtd_d_full}")
print(f"    Faixa E com aprovacao de valor solicitado: {qtd_e_full}")
if pontos_atencao:
    print("  Pontos de atencao:")
    for ponto in pontos_atencao:
        print(f"    - {ponto}")
else:
    print("  Pontos de atencao: sem violacao automatica nas faixas D/E.")
print(f"{'='*65}")

  POLITICA FINAL RECOMENDADA: Expansivo com controle de risco
  Identificador tecnico: Expansivo
  Produto: Emprestimo bancario parcelado - Pessoa fisica
-----------------------------------------------------------------
  Formula:
    parcela_max = renda x pct_faixa x red_restritivo x red_inativo x red_tempo
    limite_max  = parcela_max x [(1-(1+taxa)^(-prazo))/taxa]
-----------------------------------------------------------------
  A - Baixo risco             : 45%
  B - Médio-baixo risco       : 40%
  C - Médio risco             : 35%
  D - Alto risco              : 25%
  E - Muito alto risco        : -> Recusar
-----------------------------------------------------------------
  Redutor cliente inativo: 90%
  Redutores restritivos (sem/2/5/10/>10%): 100%/95%/85%/70%/50%
  Redutores tempo conta (curto/medio/longo): 92%/98%/100%
  Valor minimo para aprovacao reduzida: R$ 500
-----------------------------------------------------------------
  Resultados historicos (backtest na safra d

## 12. Limitações, riscos e próximos passos

### 12.1 Limitações metodológicas

| # | Limitação | Implicação |
|---|---|---|
| 1 | Base contém apenas operações concedidas | Viés de seleção: clientes recusados historicamente não aparecem |
| 2 | Não há propostas recusadas | Impossível calibrar política com desempenho de recusas |
| 3 | Não há informação de garantia | Política mais conservadora por ausência de mitigador de perda |
| 4 | Não há LGD | Não é possível calcular perda esperada formal |
| 5 | Não há EAD regulatório | Valor emprestado usado como proxy de exposição |
| 6 | Não há score externo de bureau | `pd_score` é rating interno — não reflete visão sistêmica |
| 7 | Não há histórico detalhado de atrasos | Restritivos são proxy indireto e limitado |
| 8 | Idade e escolaridade exigem governança | Risco de viés; não usadas diretamente nas regras |
| 9 | Validação apenas em backtest histórico | Não equivale a experimento controlado ou piloto |
| 10 | Intervalos de PD dependem da calibração atual | Recalibrar se o modelo for retreinado |

### 12.2 Riscos e monitoramento

- **Estabilidade do score:** monitorar PSI de `pd_score` mensalmente
- **Safras futuras:** acompanhar bad rate por faixa de risco e por decisão
- **Fairness:** verificar se algum grupo (agência, faixa etária) está sendo sistematicamente recusado
- **Capacidade operacional:** garantir que o volume de análise manual é absorvível
- **Deriva do modelo:** recalibrar se KS ou AUC caírem materialmente em safras futuras

### 12.3 Próximos passos

1. Validar parâmetros com a área de política de crédito da instituição
2. Definir apetite de risco formal (bad rate máximo aceitável)
3. Incluir propostas recusadas para análise de population stability
4. Avaliar inclusão de score externo de bureau quando disponível
5. Implementar monitoramento contínuo por safra com alertas automáticos
6. Conduzir análise de fairness por grupos protegidos antes de produção
7. Definir processo de revisão periódica da política (mínimo semestral)

## 13. Conclusão do notebook

### O que foi feito

Este notebook comparou três cenários de política de concessão de empréstimo bancário parcelado (Conservador, Equilibrado e Expansivo) e recomendou uma política final baseada em evidência histórica por meio de score gerencial ponderado.

### Como o score funciona

O `pd_score` foi criado por modelagem supervisionada de Probabilidade de Default (PD) com split temporal no notebook 04. Ele funciona como **rating interno de risco** e não é score externo de bureau. A política usa esse score como primeira camada de triagem, mas a decisão final combina:

1. **Risco** - faixa A a E definida pelo `pd_score`
2. **Capacidade de pagamento** - `valor_renda` como base do limite máximo
3. **Restritivos** - `restritivos_sobre_renda` como redutor multiplicativo
4. **Relacionamento** - `flag_cliente_ativo` e `tempo_conta_anos` como redutores adicionais
5. **Operação** - `valor_taxa` e `valor_prazo` na conversão de parcela máxima em limite via VP

A sequência consolidada da decisão é: `PD score -> faixa de risco -> capacidade de pagamento -> redutores -> valor máximo sugerido -> decisão final`.

### O que os cenários mostram

Os três cenários revelam um **trade-off estrutural**: cenários mais conservadores reduzem inadimplência dos aprovados ao custo de menor aprovação e exposição; cenários mais expansivos aumentam volume ao custo de maior risco. Não existe um cenário dominante - a escolha depende do **apetite de risco** da instituição.

### Cenário recomendado

O identificador técnico do cenário recomendado é `Expansivo`. Para apresentação gerencial, ele deve ser comunicado como **Expansivo com controle de risco**.

A recomendação se justifica porque esse cenário preserva mais aprovação automática e exposição financeira, mas mantém a inadimplência dos aprovados em torno de **6,74%**, abaixo da taxa histórica de inadimplência de **12,30%**.

---

> **Importante:** Esta é uma **política inicial simulada**, baseada em backtest histórico sobre uma base de operações concedidas. Ela deve ser **validada pela área de política de crédito** da instituição, calibrada conforme o apetite de risco formal e monitorada continuamente por safra após implantação.

In [28]:
# Checklist de saídas geradas
saidas = [
    ("data/processed", "base_simulacao_cenarios_politica.parquet"),
    ("outputs/tables", "politica_contrato_dados_cenarios.csv"),
    ("outputs/tables", "politica_variaveis_disponiveis_limitacoes.csv"),
    ("outputs/tables", "politica_parametros_cenarios.csv"),
    ("outputs/tables", "politica_resumo_cenarios.csv"),
    ("outputs/tables", "politica_resumo_decisoes_cenarios.csv"),
    ("outputs/tables", "politica_resumo_faixas_cenarios.csv"),
    ("outputs/tables", "politica_score_por_decisao_cenarios.csv"),
    ("outputs/tables", "politica_score_gerencial_cenarios.csv"),
    ("outputs/tables", "politica_tabela_executiva_recomendacao.csv"),
    ("outputs/tables", "politica_estrutura_consolidada_decisao.csv"),
    ("outputs/tables", "politica_final_recomendada.csv"),
]

print("Checklist de saídas do notebook 06:")
print(f"{'-'*58}")
todos_ok = True
for diretorio, arquivo in saidas:
    caminho = ROOT / diretorio / arquivo
    existe = caminho.exists()
    if not existe:
        todos_ok = False
    print(f"  {'OK' if existe else 'XX'}  {diretorio}/{arquivo}")

print(f"{'-'*58}")
status_final = "Todas as saídas foram geradas." if todos_ok else "ATENÇÃO: algumas saídas estão ausentes."
print(status_final)
print()
print(f"Cenário recomendado: {CENARIO_RECOMENDADO}")
print(f"Nome gerencial: {NOME_GERENCIAL_RECOMENDADO}")

Checklist de saídas do notebook 06:
----------------------------------------------------------
  OK  data/processed/base_simulacao_cenarios_politica.parquet
  OK  outputs/tables/politica_contrato_dados_cenarios.csv
  OK  outputs/tables/politica_variaveis_disponiveis_limitacoes.csv
  OK  outputs/tables/politica_parametros_cenarios.csv
  OK  outputs/tables/politica_resumo_cenarios.csv
  OK  outputs/tables/politica_resumo_decisoes_cenarios.csv
  OK  outputs/tables/politica_resumo_faixas_cenarios.csv
  OK  outputs/tables/politica_score_por_decisao_cenarios.csv
  OK  outputs/tables/politica_score_gerencial_cenarios.csv
  OK  outputs/tables/politica_tabela_executiva_recomendacao.csv
  OK  outputs/tables/politica_estrutura_consolidada_decisao.csv
  OK  outputs/tables/politica_final_recomendada.csv
----------------------------------------------------------
Todas as saídas foram geradas.

Cenário recomendado: Expansivo
Nome gerencial: Expansivo com controle de risco
